# Setup

In [ ]:
from pathlib import Path
import xarray as xr
import pandas as pd
import numpy as np

# Path to the NetCDF file
NC_FILE = Path("../local_only/1989.monthly_rain.nc")

# NetCDFViewer Class

In [ ]:
class NetCDFViewer:
    """Context manager for exploring and extracting data from NetCDF files."""
    
    def __init__(self, path):
        """Initialize with a path to a NetCDF file (doesn't open until context manager entry)."""
        self.path = Path(path)
        self.ds = None
    
    def __enter__(self):
        """Open the dataset on context manager entry."""
        self.ds = xr.open_dataset(self.path, engine="netcdf4")
        return self
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        """Close the dataset on context manager exit."""
        if self.ds is not None:
            self.ds.close()
        return False
    
    def summary(self):
        """Print structured summary of the NetCDF file."""
        if self.ds is None:
            raise RuntimeError("Dataset not open. Use 'with NetCDFViewer(path) as viewer:'")
        
        print(f"File: {self.path.resolve()}\n")
        
        # Dimensions
        print("Dimensions:")
        for dim_name, dim_size in self.ds.dims.items():
            print(f"  {dim_name:<20} {dim_size}")
        
        # Variables
        print("\nVariables:")
        for var_name, var in self.ds.data_vars.items():
            units = var.attrs.get("units", "—")
            long_name = var.attrs.get("long_name", "—")
            print(f"  {var_name:<20} shape={str(var.shape):<20} dtype={str(var.dtype):<10}")
            print(f"    units: {units}")
            print(f"    long_name: {long_name}")
        
        # Global Attributes
        print("\nGlobal Attributes:")
        if self.ds.attrs:
            for attr_name, attr_value in self.ds.attrs.items():
                # Truncate very long attribute values
                attr_str = str(attr_value)[:80]
                if len(str(attr_value)) > 80:
                    attr_str += "..."
                print(f"  {attr_name:<30} {attr_str}")
        else:
            print("  (none)")
    
    def coordinates(self):
        """Print coordinate information (lat/lon or similar)."""
        if self.ds is None:
            raise RuntimeError("Dataset not open. Use 'with NetCDFViewer(path) as viewer:'")
        
        # Try common latitude naming patterns
        lat_name = None
        for name in ["lat", "latitude", "y"]:
            if name in self.ds.coords:
                lat_name = name
                break
        
        # Try common longitude naming patterns
        lon_name = None
        for name in ["lon", "longitude", "x"]:
            if name in self.ds.coords:
                lon_name = name
                break
        
        print("Coordinates:")
        if lat_name:
            lat_values = self.ds.coords[lat_name].values
            print(f"  {lat_name:<20} {len(lat_values):>6} values  min={lat_values.min():.4f}  max={lat_values.max():.4f}")
        else:
            print("  latitude: not found (tried lat, latitude, y)")
        
        if lon_name:
            lon_values = self.ds.coords[lon_name].values
            print(f"  {lon_name:<20} {len(lon_values):>6} values  min={lon_values.min():.4f}  max={lon_values.max():.4f}")
        else:
            print("  longitude: not found (tried lon, longitude, x)")
    
    def get_spatial_slice(self, time, variable=None):
        """Extract a spatial slice for a given time step.
        
        Args:
            time: ISO string (e.g. "1989-06", "1989-06-15", "1989-06-15 14:30")
            variable: Variable name. If None, auto-selects if only one real data variable exists.
        
        Returns:
            xarray.DataArray: 2D spatial grid for the given time.
        """
        if self.ds is None:
            raise RuntimeError("Dataset not open. Use 'with NetCDFViewer(path) as viewer:'")
        
        # Auto-select variable if not specified
        if variable is None:
            # Filter out non-numeric/scalar variables (e.g., crs)
            numeric_vars = [
                name for name, var in self.ds.data_vars.items()
                if var.dtype not in [object, str] and len(var.dims) > 0
            ]
            
            if len(numeric_vars) == 0:
                raise ValueError("No numeric data variables found in the dataset.")
            elif len(numeric_vars) == 1:
                variable = numeric_vars[0]
            else:
                raise ValueError(
                    f"Multiple data variables found: {numeric_vars}. "
                    f"Please specify which one to extract."
                )
        
        # Select by time (nearest neighbour)
        sliced = self.ds[variable].sel(time=time, method="nearest")
        
        return sliced

# Explore the Dataset

In [ ]:
with NetCDFViewer(NC_FILE) as viewer:
    viewer.summary()

In [ ]:
with NetCDFViewer(NC_FILE) as viewer:
    viewer.coordinates()

# Extract a Spatial Slice

In [ ]:
# Extract rainfall data for June 1989
with NetCDFViewer(NC_FILE) as viewer:
    june_rainfall = viewer.get_spatial_slice("1989-06")
    print(f"Shape: {june_rainfall.shape}")
    print(f"Data type: {june_rainfall.dtype}")
    print(f"Min: {june_rainfall.min().item():.2f}, Max: {june_rainfall.max().item():.2f}")
    print(f"\nSelected time: {june_rainfall.coords['time'].values}")